# MahaMonsoon-X — AIMRI Rainfall Baseline
### Adaptive / historical rainfall mean • Monthly gridded ERA5 data

This notebook implements **AIMRI only** as a transparent, provider-independent statistical baseline. It does not train Linear Regression, ConvLSTM, or another machine-learning estimator.

The baseline uses historical precipitation at the same grid cell and same calendar month to predict future precipitation. Its main hyperparameters are:
- **lookback window:** number of previous years included;
- **weighting strategy:** equal weights or recency-weighted averages;
- **minimum history:** how many valid historical observations are required.

The notebook selects settings using a chronological validation period and evaluates the selected configuration on a later held-out test period. It reads the already-preprocessed `data/processed/era5_processed.csv`.

> **Naming note:** AIMRI is treated here as the project’s historical-month rainfall baseline. If AIMRI has a formally defined expansion or an approved formula in your project specification, replace the definition below with that exact specification. The implementation is explicit so it can be audited.

## 1. Mathematical definition

Let \(R_{y,m,i,j}\) denote observed precipitation for year \(y\), calendar month \(m\), and grid cell \((i,j)\). For a lookback of \(K\) years, the equal-weight AIMRI estimate is

\[
\widehat{R}^{\mathrm{AIMRI}}_{y,m,i,j}
=\frac{1}{K}\sum_{k=1}^{K}R_{y-k,m,i,j}.
\]

This uses the same calendar month and exact same grid cell in the preceding years. For example, the July 2020 estimate with \(K=5\) averages July 2015–2019 at that cell.

### Recency-weighted option

A recency-weighted variant gives more influence to recent years:

\[
\widehat{R}^{\mathrm{AIMRI}}_{y,m,i,j}
=\frac{\sum_{k=1}^{K} w_k R_{y-k,m,i,j}}
{\sum_{k=1}^{K}w_k},
\qquad
w_k=\rho^{k-1}, \quad 0<\rho\leq 1.
\]

Here \(k=1\) is the most recent year, so its weight is 1; older years receive progressively smaller weights when \(\rho<1\). When \(\rho=1\), this becomes the equal-weight mean.

### Eligibility and missing data

Only finite, observed historical rainfall values contribute to a prediction. The denominator is the sum of weights for available values. A prediction is emitted only if at least `MIN_HISTORY` of the requested previous-year values are present. Missing target observations are not interpreted as zero rainfall.

### Evaluation metrics

\[
\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\widehat{y}_i|,
\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat{y}_i)^2}.
\]

\[
R^2=1-\frac{\sum_i(y_i-\widehat{y}_i)^2}
{\sum_i(y_i-\bar{y})^2}.
\]

Lower MAE/RMSE is better. \(R^2\) can be negative. These are regression metrics, not classification accuracy. All metrics use the processed target scale; if `tp` was standardized upstream, they are standardized units rather than physical rainfall units.

## 2. Imports, paths, and reproducibility

Run from the project root. Outputs are isolated under `artifacts/aimri/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().parent

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "aimri"

PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "era5_processed.csv"

MODEL_DIR = ARTIFACT_ROOT / "models"
RESULTS_DIR = ARTIFACT_ROOT / "results"
FIGURE_DIR = ARTIFACT_ROOT / "figures"


for folder in (MODEL_DIR, RESULTS_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH.resolve()}. Run from the project root or update CSV_PATH."
    )
print("Input:", CSV_PATH.resolve())
print("Output:", ARTIFACT_ROOT.resolve())

## 3. Load the processed dataset

The CSV uses `timestamp`, `latitude`, `longitude`, and `tp`. Year and month are extracted from the timestamp. No new scaling or imputation is performed here.

In [ ]:
data = pd.read_csv(CSV_PATH)
data.columns = [str(c).strip() for c in data.columns]

required = ["timestamp", "latitude", "longitude", "tp"]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

data["timestamp"] = pd.to_datetime(data["timestamp"], errors="raise")
data["year"] = data["timestamp"].dt.year.astype(int)
data["month"] = data["timestamp"].dt.month.astype(int)

for col in ["latitude", "longitude", "tp"]:
    data[col] = pd.to_numeric(data[col], errors="raise")

# AIMRI needs only time, coordinates, and the rainfall target.
data = data[["timestamp", "year", "month", "latitude", "longitude", "tp"]].copy()
data = data.replace([np.inf, -np.inf], np.nan)

# One value per year-month-grid key; average duplicates if any exist.
key_cols = ["year", "month", "latitude", "longitude"]
duplicate_count = int(data.duplicated(key_cols).sum())
if duplicate_count:
    data = data.groupby(key_cols, as_index=False).agg(
        timestamp=("timestamp", "first"),
        tp=("tp", "mean")
    )
    data["timestamp"] = pd.to_datetime(
        dict(year=data["year"], month=data["month"], day=1)
    )

print("Data shape:", data.shape)
print("Date range:", data["timestamp"].min(), "to", data["timestamp"].max())
print("Grid cells:", data[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Year range:", data["year"].min(), "to", data["year"].max())
print("Duplicate keys resolved:", duplicate_count)
display(data.head())


## 4. Define chronological train / validation / test periods

The split is based on target year:
- oldest years: training/reference period;
- next years: validation for selecting lookback and recency weighting;
- latest years: final test period.

AIMRI does not fit coefficients, but its historical reference observations must still be restricted to dates before each target month. Test-year observations are used only as past history when they precede a later test target, which matches an operational rolling forecast. The target month itself is never used in its own prediction.

In [ ]:
years = np.sort(data["year"].unique())
if len(years) < 8:
    raise ValueError(
        f"Only {len(years)} unique years found. AIMRI tuning needs a longer historical record."
    )

n_years = len(years)
n_train = max(5, int(n_years * 0.70))
n_val = max(1, int(n_years * 0.15))
if n_train + n_val >= n_years:
    n_train = n_years - 2
    n_val = 1

train_years = years[:n_train]
val_years = years[n_train:n_train+n_val]
test_years = years[n_train+n_val:]

print(f"Reference/train years: {train_years[0]}–{train_years[-1]}")
print(f"Validation target years: {val_years[0]}–{val_years[-1]}")
print(f"Test target years: {test_years[0]}–{test_years[-1]}")


## 5. Build a fast year-month-grid lookup

The implementation uses vectorized joins, not repeated row-wise searches. For each target year/month/grid cell, the table contains historical rainfall values at year offsets 1 through the maximum lookback. This makes hyperparameter tuning efficient and auditable.

In [ ]:
MAX_LOOKBACK = 10
MIN_HISTORY = 3

# Build a lookup table keyed by (year, month, latitude, longitude).
history = data[["year", "month", "latitude", "longitude", "tp"]].copy()
history = history.rename(columns={"tp": "rainfall"})
base = data[["year", "month", "latitude", "longitude", "tp"]].copy()
base = base.rename(columns={"tp": "actual"})

for lag in range(1, MAX_LOOKBACK + 1):
    shifted = history.copy()
    shifted["year"] = shifted["year"] + lag
    shifted = shifted.rename(columns={"rainfall": f"lag_{lag}"})
    base = base.merge(
        shifted[["year", "month", "latitude", "longitude", f"lag_{lag}"]],
        on=["year", "month", "latitude", "longitude"],
        how="left",
        validate="one_to_one"
    )

lag_columns = [f"lag_{lag}" for lag in range(1, MAX_LOOKBACK + 1)]
print("Lookup table shape:", base.shape)
display(base.head())


## 6. AIMRI hyperparameter tuning

We tune three choices:
- lookback \(K \in \{3,5,7,10\}\) years (limited by `MAX_LOOKBACK`);
- recency factor \(\rho \in \{1.0,0.8,0.6\}\);
- minimum valid historical years `MIN_HISTORY` fixed to 3 for a fair, interpretable comparison.

The equal-weight baseline is included as \(\rho=1\). Hyperparameters are selected by validation MAE, then the selected setting is evaluated on the test years. The validation period is not used as the final test.

In [ ]:
LOOKBACK_CANDIDATES = [k for k in [3, 5, 7, 10] if k <= MAX_LOOKBACK]
RECENCY_CANDIDATES = [1.0, 0.8, 0.6]
MIN_HISTORY = 3

def aimri_predict(frame, lookback, recency, min_history=MIN_HISTORY):
    cols = [f"lag_{k}" for k in range(1, lookback + 1)]
    values = frame[cols].to_numpy(dtype=float)

    # Lag 1 is most recent, lag 2 is next most recent, and so on.
    weights = np.array([recency ** (k - 1) for k in range(1, lookback + 1)], dtype=float)
    valid = np.isfinite(values)
    weighted_sum = np.nansum(values * weights[None, :], axis=1)
    available_weight = (valid * weights[None, :]).sum(axis=1)
    valid_count = valid.sum(axis=1)

    prediction = np.divide(
        weighted_sum,
        available_weight,
        out=np.full(len(frame), np.nan, dtype=float),
        where=(available_weight > 0) & (valid_count >= min_history)
    )
    return prediction

def score_predictions(frame, predictions, label):
    actual = frame["actual"].to_numpy(dtype=float)
    valid = np.isfinite(actual) & np.isfinite(predictions)
    y_true, y_pred = actual[valid], predictions[valid]
    if len(y_true) == 0:
        return {"model": label, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "n": 0}
    return {
        "model": label,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "n": int(len(y_true))
    }

validation_frame = base[base["year"].isin(val_years)].copy()
tuning_rows = []
best_config, best_val_mae = None, float("inf")

for lookback in LOOKBACK_CANDIDATES:
    for recency in RECENCY_CANDIDATES:
        predictions = aimri_predict(validation_frame, lookback, recency)
        row = score_predictions(
            validation_frame, predictions,
            f"AIMRI K={lookback}, rho={recency}"
        )
        row.update({"lookback_years": lookback, "recency_factor": recency})
        tuning_rows.append(row)
        if np.isfinite(row["MAE"]) and row["MAE"] < best_val_mae:
            best_val_mae = row["MAE"]
            best_config = {
                "lookback_years": lookback,
                "recency_factor": recency,
                "min_history": MIN_HISTORY
            }

tuning_results = pd.DataFrame(tuning_rows).sort_values("MAE")
display(tuning_results)
tuning_results.to_csv(RESULTS_DIR / "hyperparameter_tuning.csv", index=False)

if best_config is None:
    raise ValueError(
        "No valid AIMRI validation predictions. Check historical coverage and missing target values."
    )
print("Selected AIMRI configuration:", best_config)
print("Best validation MAE:", best_val_mae)


## 7. Evaluate the selected AIMRI configuration on the held-out test years

For a target in year \(y\), lag \(k\) uses the observation from year \(y-k\), same calendar month and same grid cell. Thus, every prediction uses only historical values relative to its own target year. The first test year can use earlier training/validation years; later test years can use prior test-year observations, as would a rolling annual forecast.

In [ ]:
test_frame = base[base["year"].isin(test_years)].copy()
test_predictions = aimri_predict(
    test_frame,
    lookback=best_config["lookback_years"],
    recency=best_config["recency_factor"],
    min_history=best_config["min_history"]
)

test_metrics_row = score_predictions(test_frame, test_predictions, "AIMRI")
test_metrics = pd.DataFrame([{
    **test_metrics_row,
    "lookback_years": best_config["lookback_years"],
    "recency_factor": best_config["recency_factor"],
    "test_year_start": int(test_years[0]),
    "test_year_end": int(test_years[-1])
}])
display(test_metrics.round(6))
test_metrics.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)

comparison = test_frame[
    ["year", "month", "latitude", "longitude", "actual"]
].copy()
comparison["aimri_prediction"] = test_predictions
comparison["residual_actual_minus_prediction"] = (
    comparison["actual"] - comparison["aimri_prediction"]
)
comparison.to_csv(RESULTS_DIR / "test_grid_predictions.csv", index=False)
display(comparison.dropna(subset=["aimri_prediction"]).head())


## 8. Visual diagnostics

Plots are saved to `artifacts/aimri/figures/`. The sampled line plot is a diagnostic ordering of grid-month observations, not a continuous time series for a single location.

In [ ]:
# Hyperparameter search: validation MAE.
pivot_mae = tuning_results.pivot(
    index="lookback_years", columns="recency_factor", values="MAE"
).sort_index()
plt.figure(figsize=(8, 4.5))
for recency in pivot_mae.columns:
    plt.plot(pivot_mae.index, pivot_mae[recency], marker="o", label=f"rho={recency}")
plt.xlabel("Lookback years")
plt.ylabel("Validation MAE")
plt.title("AIMRI Hyperparameter Tuning")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "validation_mae_tuning.png", dpi=160, bbox_inches="tight")
plt.show()

valid_comparison = comparison.dropna(subset=["aimri_prediction"]).copy()
sample = valid_comparison.sample(
    n=min(1500, len(valid_comparison)), random_state=SEED
).sort_values(["year", "month", "latitude", "longitude"])

plt.figure(figsize=(13, 5))
plt.plot(sample["actual"].to_numpy(), label="Actual", linewidth=1)
plt.plot(sample["aimri_prediction"].to_numpy(), label="AIMRI", linewidth=1)
plt.xlabel("Sampled held-out grid-month observations")
plt.ylabel("Processed tp units")
plt.title("AIMRI: Held-out Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_prediction_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(valid_comparison["actual"], valid_comparison["aimri_prediction"], s=8, alpha=0.3)
low = float(min(valid_comparison["actual"].min(), valid_comparison["aimri_prediction"].min()))
high = float(max(valid_comparison["actual"].max(), valid_comparison["aimri_prediction"].max()))
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual processed tp")
plt.ylabel("AIMRI prediction")
plt.title("AIMRI Actual vs Predicted")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_actual_vs_predicted.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4.5))
residuals = valid_comparison["residual_actual_minus_prediction"].to_numpy()
plt.hist(residuals, bins=50)
plt.axvline(0, linewidth=1)
plt.xlabel("Residual (actual − prediction)")
plt.ylabel("Grid-month count")
plt.title("AIMRI Residual Distribution")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "test_residual_distribution.png", dpi=160, bbox_inches="tight")
plt.show()


## 9. Save AIMRI configuration and reproducibility metadata

AIMRI is a deterministic statistical rule, not a fitted neural network. Its “model artifact” is the selected configuration and metadata describing how to reproduce its predictions. The original input CSV is not copied or modified.

In [ ]:
config_path = MODEL_DIR / "aimri_config.json"
metadata_path = MODEL_DIR / "model_metadata.json"

config_path.write_text(json.dumps(best_config, indent=2), encoding="utf-8")
metadata = {
    "project": "MahaMonsoon-X",
    "model": "AIMRI historical same-month same-grid rainfall baseline",
    "input_csv": str(CSV_PATH),
    "target_column": "tp",
    "key_columns": ["year", "month", "latitude", "longitude"],
    "formula": "weighted mean of previous K same-month values at the same grid cell",
    "lag_1_is": "most recent previous year",
    "hyperparameter_search": {
        "lookback_candidates": LOOKBACK_CANDIDATES,
        "recency_candidates": RECENCY_CANDIDATES,
        "minimum_history": MIN_HISTORY
    },
    "selected_configuration": best_config,
    "train_years": [int(train_years[0]), int(train_years[-1])],
    "validation_years": [int(val_years[0]), int(val_years[-1])],
    "test_years": [int(test_years[0]), int(test_years[-1])],
    "test_metrics": test_metrics.to_dict(orient="records"),
    "target_units_note": "Uses processed CSV target scale; verify upstream standardization and ERA5 units."
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved artifacts:")
for path in [
    config_path, metadata_path,
    RESULTS_DIR / "hyperparameter_tuning.csv",
    RESULTS_DIR / "test_metrics.csv",
    RESULTS_DIR / "test_grid_predictions.csv"
]:
    print(f" - {path.resolve()} ({path.stat().st_size / (1024**2):.2f} MiB)")


## 10. Reload configuration and verify reproducibility

Reload the saved AIMRI configuration and confirm it reproduces the predictions from the selected configuration. This is the equivalent of a model reload test for a deterministic baseline.

In [ ]:
with config_path.open("r", encoding="utf-8") as file:
    loaded_config = json.load(file)

reloaded_predictions = aimri_predict(
    test_frame,
    lookback=int(loaded_config["lookback_years"]),
    recency=float(loaded_config["recency_factor"]),
    min_history=int(loaded_config["min_history"])
)
valid_reload = np.isfinite(test_predictions) & np.isfinite(reloaded_predictions)
assert np.allclose(
    test_predictions[valid_reload],
    reloaded_predictions[valid_reload],
    atol=1e-12, rtol=1e-10
)
print("AIMRI reload verification passed.")
print("Predictions reproduced:", int(valid_reload.sum()))


## 11. Output inventory and interpretation checklist

```text
artifacts/aimri/
├── models/
│   ├── aimri_config.json
│   └── model_metadata.json
├── results/
│   ├── hyperparameter_tuning.csv
│   ├── test_metrics.csv
│   └── test_grid_predictions.csv
└── figures/
    ├── validation_mae_tuning.png
    ├── test_prediction_comparison.png
    ├── test_actual_vs_predicted.png
    └── test_residual_distribution.png
```

Before interpreting performance, verify the precipitation units and any upstream target standardization. This AIMRI implementation is a historical baseline; it does not use weather covariates, learn neural-network weights, or claim physical causal relationships. It predicts a monthly value for each grid cell separately.